## BIS ADVANCED

### Archivo matriz espectral

## Preparado

In [6]:
import os
import glob
import shutil
import pandas as pd

In [7]:
def localizar_archivos_fa(ruta_raiz):
    """
    Busca archivos .f_a dentro de subcarpetas cuyo nombre empiece por DH.
    """
    patron = os.path.join(ruta_raiz, "**", "DH*", "*.f_a")
    archivos = glob.glob(patron, recursive=True)
    print(f"Se han encontrado {len(archivos)} archivos .f_a")
    return archivos


def clonar_a_csv(ruta_fa):
    """
    Copia el archivo .f_a y crea una versión .csv con el mismo contenido.
    """
    ruta_csv = os.path.splitext(ruta_fa)[0] + ".csv"
    shutil.copy2(ruta_fa, ruta_csv)
    print(f"Archivo copiado como: {ruta_csv}")
    return ruta_csv


def procesar_datos_csv(ruta_csv):
    """
    Procesa el archivo BIS:
    - Lee el archivo usando '|' como separador principal
    - Se queda con las columnas Time y Spectra
    - Divide Spectra en 60 columnas
    - Convierte Time a datetime
    - Convierte Spectra a float
    - Divide los valores entre 100
    - Guarda un archivo procesado con sufijo _proc
    """
    try:
        # Leer archivo
        df = pd.read_csv(
            ruta_csv,
            sep="|",
            header=None,
            skiprows=2,
            engine="python"
        )

        # Eliminar columnas completamente vacías
        df = df.dropna(axis=1, how="all")

        # Quedarnos solo con las dos columnas reales
        df = df.iloc[:, :2]
        df.columns = ["Time", "Spectra"]

        # Dividir columna Spectra en varias columnas
        spectra = df["Spectra"].astype(str).str.split(",", expand=True)
        spectra.columns = [f"Spectra_{i+1}" for i in range(spectra.shape[1])]

        # Unir Time y espectros
        df_final = pd.concat([df[["Time"]], spectra], axis=1)

        # Convertir tipos
        df_final["Time"] = pd.to_datetime(
            df_final["Time"],
            format="%m/%d/%Y %H:%M:%S"
        )
        df_final.iloc[:, 1:] = df_final.iloc[:, 1:].astype(float) / 100

        # Guardar resultado
        base, ext = os.path.splitext(ruta_csv)
        ruta_salida = f"{base}_proc{ext}"
        df_final.to_csv(ruta_salida, index=False)

        print(f"Archivo procesado guardado en: {ruta_salida}")
        return df_final

    except Exception as e:
        print(f"Error en el procesamiento de {ruta_csv}: {e}")
        return None


def procesar_todos_los_archivos(ruta_base):
    """
    Ejecuta todo el flujo sobre todos los archivos .f_a encontrados.
    """
    archivos_fa = localizar_archivos_fa(ruta_base)

    resultados = {}

    for ruta_fa in archivos_fa:
        print(f"\nProcesando: {ruta_fa}")
        ruta_csv = clonar_a_csv(ruta_fa)
        df_final = procesar_datos_csv(ruta_csv)

        if df_final is not None:
            resultados[ruta_fa] = df_final

    return resultados

In [8]:
ruta_base = "./data_bis_advanced"
resultados = procesar_todos_los_archivos(ruta_base)

Se han encontrado 2 archivos .f_a

Procesando: ./data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.f_a
Archivo copiado como: ./data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035.csv
Archivo procesado guardado en: ./data_bis_advanced\M-TA6m-03041035\DH03041035\L03041035_proc.csv

Procesando: ./data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.f_a
Archivo copiado como: ./data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035.csv
Archivo procesado guardado en: ./data_bis_advanced\M-TA6m-03041035_2\DH03041035\L03041035_proc.csv


In [9]:
# probar ver como se ve finalmente el archivo

primer_archivo = list(resultados.keys())[0]
display(resultados[primer_archivo].head())
print(resultados[primer_archivo].shape)

,Time,Spectra_1,Spectra_2,Spectra_3,Spectra_4,Spectra_5,Spectra_6,Spectra_7,Spectra_8,Spectra_9,...,Spectra_51,Spectra_52,Spectra_53,Spectra_54,Spectra_55,Spectra_56,Spectra_57,Spectra_58,Spectra_59,Spectra_60
0,2026-03-04 10:35:21,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
1,2026-03-04 10:35:22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
2,2026-03-04 10:35:23,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
3,2026-03-04 10:35:24,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
4,2026-03-04 10:35:25,93.56,98.12,97.43,100.48,99.82,96.09,93.46,98.54,101.14,...,69.31,75.63,78.86,82.6,84.23,82.09,77.32,79.75,80.81,76.02


(2119, 61)


In [10]:
# el segundo era una copia del 1

seg_archivo = list(resultados.keys())[1]
display(resultados[seg_archivo].head())
print(resultados[seg_archivo].shape)

,Time,Spectra_1,Spectra_2,Spectra_3,Spectra_4,Spectra_5,Spectra_6,Spectra_7,Spectra_8,Spectra_9,...,Spectra_51,Spectra_52,Spectra_53,Spectra_54,Spectra_55,Spectra_56,Spectra_57,Spectra_58,Spectra_59,Spectra_60
0,2026-03-04 10:35:21,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
1,2026-03-04 10:35:22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
2,2026-03-04 10:35:23,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
3,2026-03-04 10:35:24,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00
4,2026-03-04 10:35:25,93.56,98.12,97.43,100.48,99.82,96.09,93.46,98.54,101.14,...,69.31,75.63,78.86,82.6,84.23,82.09,77.32,79.75,80.81,76.02


(2119, 61)


## BIS ANTIGUO